# Xử lý & chuẩn hóa nhãn L1 / L2 / L3

Dựa trên phân tích 3 biểu đồ phân phối nhãn từ 5000 bài VnExpress.

**Pipeline:**
```
Cell 1  → Load data + kiểm tra nhanh
Cell 2  → Normalize chữ hoa/thường, khoảng trắng
Cell 3  → Hardcode MERGE_MAP: gộp/loại nhãn L2
Cell 4  → Hardcode MERGE_MAP: gộp/loại nhãn L3
Cell 5  → Xử lý đặc biệt: tách 'Tin tức' theo domain cha
Cell 6  → Áp dụng toàn bộ mapping vào 5000 bài
Cell 7  → Thống kê trước/sau để kiểm tra
Cell 8  → Build HIERARCHY từ data đã sạch
Cell 9  → Encode multi-hot + build vocab + lưu file
Cell 10 → Kiểm tra output cuối
```


## Cell 1 — Load & kiểm tra nhanh


In [1]:
# ============================================================
# CELL 1 — Load data, xem cấu trúc, đếm sơ bộ
# ============================================================
import json, re
from collections import Counter, defaultdict
from pathlib import Path

with open('raw_data.json', encoding='utf-8') as f:
    raw = json.load(f)

print(f'Tổng bài: {len(raw)}')
print(f'Keys: {list(raw[0].keys())}')
print(f'\nMẫu 2 bài:')
for a in raw[:2]:
    print(f'  L1={a.get("label_l1","")!r:20} L2={a.get("label_l2","")!r:20} L3={a.get("label_l3","")!r}')

# Đếm nhanh tình trạng nhãn
n_l1 = sum(1 for a in raw if a.get('label_l1','').strip())
n_l2 = sum(1 for a in raw if a.get('label_l2','').strip())
n_l3 = sum(1 for a in raw if a.get('label_l3','').strip())
print(f'\nCó label_l1: {n_l1} ({n_l1/len(raw)*100:.0f}%)')
print(f'Có label_l2: {n_l2} ({n_l2/len(raw)*100:.0f}%)')
print(f'Có label_l3: {n_l3} ({n_l3/len(raw)*100:.0f}%)')


Tổng bài: 5546
Keys: ['url', 'title', 'content', 'label_l1', 'label_l2', 'label_l3']

Mẫu 2 bài:
  L1='Thời sự'            L2=''                   L3=''
  L1='Thời sự'            L2=''                   L3=''

Có label_l1: 5546 (100%)
Có label_l2: 4427 (80%)
Có label_l3: 1851 (33%)


## Cell 2 — Normalize nhãn
Chuẩn hóa chữ hoa/thường và khoảng trắng trước khi gộp.
Ví dụ: `cooking` → `Cooking`, `  Bóng đá ` → `Bóng đá`


In [2]:
# ============================================================
# CELL 2 — Normalize: chuẩn hóa chữ hoa/thường & khoảng trắng
# Chạy TRƯỚC tất cả các bước gộp để tránh miss do viết khác nhau
# ============================================================

# Map normalize: chữ sai → chữ đúng
# Thêm vào đây nếu phát hiện thêm trường hợp không nhất quán
NORMALIZE_MAP = {
    'cooking':               'Ẩm thực',
    'Cooking':               'Ẩm thực',
    'Khoa học công nghệ':    'Số hóa',
    'Chính trị & chính sách':'Chính trị',
    'Chính sách':            'Chính trị',
    'Sân khấu - Mỹ thuật':   'Sân khấu & Mỹ thuật',
    'meo-tu-van':            'Tư vấn',
}

def normalize_label(label: str) -> str:
    if not label:
        return ''
    label = label.strip()                     # bỏ khoảng trắng đầu/cuối
    label = re.sub(r'\s+', ' ', label)        # nhiều space → 1 space
    label = NORMALIZE_MAP.get(label, label)   # áp dụng map
    return label

# Áp dụng normalize vào tất cả bài (in-place)
for a in raw:
    a['label_l1'] = normalize_label(a.get('label_l1', ''))
    a['label_l2'] = normalize_label(a.get('label_l2', ''))
    a['label_l3'] = normalize_label(a.get('label_l3', ''))

# Kiểm tra
print('Top 20 nhãn L2 sau normalize:')
c = Counter(a['label_l2'] for a in raw if a['label_l2'])
for name, cnt in c.most_common(20):
    print(f'  {name:<30} {cnt}')


Top 20 nhãn L2 sau normalize:
  Tin tức                        488
  Bóng đá                        295
  Các bệnh                       282
  Nhịp sống                      200
  Giới sao                       179
  Chính trị                      168
  Thị trường                     164
  Điểm đến                       152
  Chuyển đổi số                  144
  Ngoại hạng Anh                 123
  Thiết bị                       107
  Quân sự                        105
  Thời trang                     104
  Tuyển sinh                     99
  Tổ ấm                          99
  Bài học sống                   94
  Giao thông                     88
  Các môn khác                   88
  Diễn đàn                       85
  AI                             74


## Cell 3 — Hardcode gộp nhãn L2
3 loại quy tắc:
- **MERGE**: nhãn A → nhãn B (gộp vào nhãn khác)
- **DROP**: loại bỏ hoàn toàn (sự kiện, format media, quá ít bài)
- **KEEP**: giữ nguyên (không cần ghi vào đây, mặc định là giữ)


In [3]:
# ============================================================
# CELL 3 — Hardcode MERGE / DROP cho nhãn L2
# Dựa trên phân tích biểu đồ label_l2
# ============================================================

# Nhãn L2 cần GỘP vào nhãn khác
L2_MERGE = {
    # Tên giải đấu → gộp vào chuyên mục bóng đá
    'Ngoại hạng Anh':       'Bóng đá',
    'V-League':             'Bóng đá',
    'Bundesliga':           'Bóng đá',
    'Champions League':     'Bóng đá',
    'La Liga':              'Bóng đá',
    'Serie A':              'Bóng đá',
    'Europa League':        'Bóng đá',

    # Trùng nghĩa domain cha hoặc quá chung
    'Chuyển đổi số':        'Số hóa',   # trùng nghĩa với domain Số hóa
    'Các môn khác':         '',         # DROP: quá chung, không rõ domain
    'Hậu trường':           'Giải sao', # thuộc Giải trí
    'Dấu chân':             'Điểm đến', # trùng nghĩa
    'Thế giới tự nhiên':    'Vũ trụ',   # cùng nhóm khoa học
    'Bắc Mỹ':               'Quốc tế',  # tên khu vực → chuyên mục
    'Phân tích':            'Tư liệu',  # tương tự nhau
    'Vĩ mô':                'Thị trường',
    'Dân sự':               'Hồ sơ phá án',
    'Sân khấu & Mỹ thuật':  'Phim',     # gộp vào giải trí
    'Cuộc sống đó đây':     'Nhịp sống',
    'Bài học sống':         'Nhịp sống',
    'Tổ ấm':                'Nhịp sống',
    'Tiêu dùng':            'Hàng hoá',
    'Diễn đàn':             'Chính trị',
}

# Nhãn L2 cần BỎ HOÀN TOÀN (sự kiện, format, tên chiến dịch, quá ít)
L2_DROP = {
    'Tin tức',              # xử lý riêng ở Cell 5
    'Video',                # format media
    'Ảnh',                  # format media
    'Podcast',              # format media
    'Trắc nghiệm',          # format nội dung
    'Vaccine',              # quá đặc thù, gộp vào Sức khỏe
    'Mekong',               # 11 bài, dưới ngưỡng
    'Xe điện',              # 10 bài, dưới ngưỡng
    'Chứng khoán',          # 11 bài, dưới ngưỡng
    'Netzero',              # topic tạm thời
    'Kỳ nguyên mới',        # khẩu hiệu
    'Quỹ hy vọng',          # tên quỹ cụ thể
    'Ebank',                # 6 bài
    'Của sổ tri thức',      # tên chương trình
    'Học tiếng Anh',        # 14 bài, quá đặc thù
    'Đổi mới sáng tạo',     # topic tạm thời
    'Việc làm',             # 14 bài
    'Bảo vệ trọn vẹn từng khoảnh khắc',  # tên chiến dịch
    'Bầu cử Đại biểu Quốc hội khóa 16',  # sự kiện
    'Dự án',                # quá chung
    'Kun Marathon',         # sự kiện cụ thể
    'Giáo dục 4.0',         # topic tạm thời
    'Người Việt 5 châu',    # 3 bài
    'Kinh tế vùng',         # 2 bài
    'Tiền của tôi',         # tên series
    'Khoa học trong nước',  # 2 bài
    'Khí mát lạnh, sống trọn ven',  # tên chiến dịch
    'Nội trợ',              # 2 bài
    'GameVerse 2026',       # sự kiện
    'Tương thuật',          # 1 bài
    'GameVerse 2025',       # sự kiện
    'esport',               # 1 bài
    'Bảo hiểm',             # 1 bài
    'Sống tinh tế trong thầm lặng',  # tên series
}

def apply_l2_rules(l1: str, l2: str) -> str:
    '''
    Áp dụng MERGE và DROP cho một nhãn L2.
    Trả về nhãn mới, hoặc '' nếu cần bỏ.
    '''
    if not l2:
        return ''
    if l2 in L2_DROP:
        return ''                       # bỏ
    merged = L2_MERGE.get(l2, l2)
    return merged                       # '' nếu MERGE trỏ vào ''

# Kiểm tra nhanh một vài trường hợp
tests = [
    ('Thể thao',  'Ngoại hạng Anh'),
    ('Số hóa',    'Chuyển đổi số'),
    ('Đời sống',  'Bài học sống'),
    ('Sức khỏe',  'Tin tức'),
    ('Giải trí',  'Hậu trường'),
    ('Khoa học',  'Vũ trụ'),
    ('Du lịch',   'Dấu chân'),
]
print('Kiểm tra L2_MERGE / L2_DROP:')
for l1, l2 in tests:
    result = apply_l2_rules(l1, l2)
    arrow = f'→ {result!r}' if result else '→ (bỏ)'
    print(f'  ({l1:<12}, {l2:<25}) {arrow}')


Kiểm tra L2_MERGE / L2_DROP:
  (Thể thao    , Ngoại hạng Anh           ) → 'Bóng đá'
  (Số hóa      , Chuyển đổi số            ) → 'Số hóa'
  (Đời sống    , Bài học sống             ) → 'Nhịp sống'
  (Sức khỏe    , Tin tức                  ) → (bỏ)
  (Giải trí    , Hậu trường               ) → 'Giải sao'
  (Khoa học    , Vũ trụ                   ) → 'Vũ trụ'
  (Du lịch     , Dấu chân                 ) → 'Điểm đến'


## Cell 4 — Hardcode gộp nhãn L3
L3 có rất nhiều nhãn chỉ 1–5 bài. Chiến lược: giữ lại những nhãn y tế/bệnh
rõ ràng và nhãn thể thao có đủ bài, loại bỏ sự kiện và format.


In [ ]:
# ============================================================
# CELL 4 — Hardcode MERGE / DROP cho nhãn L3
# Dựa trên biểu đồ label_l3
# L3 có quá nhiều nhãn nhỏ → chiến lược chính là DROP
# ============================================================

# Nhãn L3 cần GỘP
L3_MERGE = {
    # Giải trí
    'Nhịp sống số':      'Nhịp sống',
    'Sao đẹp - Sao xấu': 'Giải sao',
    'Chuyện màn ảnh':    'Phim',
    'Lăng văn':          'Nhịp sống',
    'Lăng nhạc':         'Nhạc',
    'Lăng mốt':          'Thời trang',
    'Bạn đọc viết':      'Nhịp sống',

    # Pháp luật / Tư vấn
    'Hỏi đáp':           'Tư vấn',
    'Y kiến':            'Tư vấn',      # ← sửa: Diễn đàn đã bị DROP ở L2

    # Thế giới
    'Phân tích':         'Tư liệu',
    'Việt Nam':          'Trong nước',

    # Xe
    'Tình huống':        'Kỹ năng lái',
    'Luật giao thông':   'Kỹ năng lái',
    'Chăm sóc xe':       'Kỹ năng lái',
    'Kinh nghiệm':       'Kỹ năng lái', # ← chỉ để ở MERGE, xóa khỏi DROP

    # Nghệ thuật
    'Mỹ thuật':          'Phim',         # Sân khấu & Mỹ thuật đã merge vào Phim

    # DROP thông qua MERGE trỏ về ''
    'Thị trường':        '',   # trùng tên L2
    'Giao thông':        '',   # trùng tên L2
    'Bóng đá':           '',   # trùng tên L2
    'Sản phẩm':          '',   # trùng tên L2
    'Diễn đàn':          '',   # L2 cha đã bị DROP
}

L3_DROP = {
    # Format media
    'Video', 'Ảnh',

    # Lỗi 1 đã sửa: chỉ giữ 1 dòng 'Đầu tư'
    'Đầu tư',

    # Lỗi 2 đã sửa: Champions League là L2, không phải L3
    # → không cần khai báo ở đây, Cell 3 đã gộp vào Bóng đá

    # Lỗi 3 đã sửa: KHÔNG DROP các nhãn bệnh quan trọng
    # Nhân khoa (24), Tim mạch (24), Nội tiết (21), Thần kinh (16)...
    # → giữ nguyên, đây là L3 có giá trị nhất của domain Sức khỏe

    # Lỗi 4 đã sửa: Vô/Tương thuật → DROP (L2 cha 'Các môn khác' đã bị DROP)
    'Vô thuật',
    'Tương thuật',

    # Lỗi 5 đã sửa: Kinh nghiệm chỉ ở MERGE, xóa khỏi đây

    # Sự kiện cụ thể
    'Kun Marathon Hồ Chí Minh',
    'Sene A',
    'Lớp 10',
    'Thi tốt nghiệp THPT',
    'Câu chuyện bảo hiểm',
    'Thúc đẩy Khoa học Công nghệ',
    'Cẩm nang đầu tư F0',
    'Game talk', 'VMC', 'Tin dự án',
    'Hành trình kiêu dũng',
    'Doanh nghiệp xanh', 'Doanh nhân',

    # Trùng tên L2
    'Chuyển đổi số',
    'Thiết bị',
    'Giáo dục phổ thông',
    'Các môn thể thao khác',

    # Quá ít bài
    'Đất đai', 'Golf', 'Làm đẹp', 'Tác giả',
    'Điểm sách', 'Đạt đến', 'Điểm phim',
    'Dấu ấn thương hiệu', 'Nhân sự', 'Bình luận', 'Chân dung',
    'Hà - Đáp', 'Sáng kiến', 'Sống bền vững',
    'Ứng dụng', 'Hình sự', 'Chính sách',
    'Trắc nghiệm', 'Thi trường',
    'Hiểm muộn', 'Ngân hàng', 'Bộ sưu tập',
    'Dua xe', 'Cờ vua',
}

def apply_l3_rules(l2: str, l3: str) -> str:
    if not l3:
        return ''
    if l3 in L3_DROP:
        return ''
    merged = L3_MERGE.get(l3, l3)
    return merged

# Kiểm tra
tests = [
    ('Giải sao',   'Sao đẹp - Sao xấu'),
    ('Phim',       'Chuyện màn ảnh'),
    ('Cầm lái',    'Tình huống'),
    ('Sức khỏe',   'Nhân khoa'),
    ('Bóng đá',    'Champions League'),
    ('Pháp luật',  'Video'),
]
print('Kiểm tra L3_MERGE / L3_DROP:')
for l2, l3 in tests:
    result = apply_l3_rules(l2, l3)
    arrow = f'→ {result!r}' if result else '→ (bỏ)'
    print(f'  ({l2:<15}, {l3:<25}) {arrow}')


## Cell 5 — Xử lý đặc biệt: tách nhãn 'Tin tức'
`Tin tức` xuất hiện 468 lần nhưng là tên chuyên mục con của **nhiều domain khác nhau**.
Cần tách theo domain cha để có nghĩa rõ ràng.


In [ ]:
# ============================================================
# CELL 5 — Tách 'Tin tức' theo domain cha (label_l1)
# Vì mỗi domain dùng tên 'Tin tức' cho chuyên mục con của mình
# Ta dùng cặp (l1, 'Tin tức') để quyết định
# ============================================================

# Xem 'Tin tức' đang xuất hiện ở domain nào với tần suất bao nhiêu
tin_tuc_by_domain = Counter(
    a['label_l1']
    for a in raw
    if a.get('label_l2','') == 'Tin tức' and a.get('label_l1','')
)
print('Phân phối Tin tức theo domain L1:')
for domain, cnt in tin_tuc_by_domain.most_common():
    print(f'  {domain:<20} {cnt} bài')

print()

# Quy tắc tách: (domain_l1) → tên nhãn mới
# Giữ nếu >= 20 bài, bỏ nếu quá ít
TIN_TUC_MAP = {
    'Sức khỏe':   'Tin tức sức khỏe',   # kiểm tra số bài trước khi quyết
    'Xe':         'Tin tức xe',
    'Số hóa':     'Tin tức số hóa',
    'Thể thao':   '',   # đã có Bóng đá, Tennis, Marathon → bỏ
    'Giáo dục':   '',   # đã có Tuyển sinh, Du học → bỏ
    'Kinh doanh': '',   # đã có Thị trường, Hàng hoá → bỏ
    'Thời sự':    '',   # đã có Chính trị, Giao thông → bỏ
    'Thế giới':   '',   # đã có Quốc tế, Tư liệu → bỏ
    'Du lịch':    '',   # đã có Điểm đến, Ẩm thực → bỏ
    'Đời sống':   '',   # đã có Nhịp sống → bỏ
    'Pháp luật':  '',   # đã có Hồ sơ phá án, Tư vấn → bỏ
    'Giải trí':   '',   # đã có Giải sao, Phim → bỏ
    'Khoa học':   '',   # đã có Vũ trụ → bỏ
}

def resolve_tin_tuc(l1: str) -> str:
    '''Trả về tên nhãn mới cho bài có label_l2=Tin tức'''
    return TIN_TUC_MAP.get(l1, '')   # mặc định bỏ nếu không có trong map

print('Quy tắc tách Tin tức:')
for domain, new_label in TIN_TUC_MAP.items():
    cnt = tin_tuc_by_domain.get(domain, 0)
    result = f'→ {new_label!r}' if new_label else '→ (bỏ)'
    print(f'  ({domain:<12}, Tin tức) {cnt:>4} bài  {result}')


## Cell 6 — Áp dụng toàn bộ quy tắc vào 5000 bài
Chạy tất cả mapping trên từng bài, lưu nhãn sạch vào field mới.


In [ ]:
# ============================================================
# CELL 6 — Áp dụng toàn bộ quy tắc vào từng bài
# Lưu vào field mới: clean_l1, clean_l2, clean_l3
# Giữ nguyên label_l1/l2/l3 gốc để so sánh nếu cần
# ============================================================

VALID_L1 = {
    'Thời sự', 'Thế giới', 'Kinh doanh', 'Khoa học', 'Giải trí',
    'Thể thao', 'Pháp luật', 'Giáo dục', 'Sức khỏe', 'Đời sống',
    'Du lịch', 'Số hóa', 'Xe',
}

for a in raw:
    l1 = a.get('label_l1', '')
    l2 = a.get('label_l2', '')
    l3 = a.get('label_l3', '')

    # Bước 1: validate L1
    clean_l1 = l1 if l1 in VALID_L1 else ''

    # Bước 2: xử lý L2
    if l2 == 'Tin tức':
        clean_l2 = resolve_tin_tuc(clean_l1)  # tách theo domain
    else:
        clean_l2 = apply_l2_rules(clean_l1, l2)

    # Bước 3: xử lý L3
    clean_l3 = apply_l3_rules(clean_l2, l3)

    # Bước 4: enforce parent-child
    # Nếu sau khi gộp, clean_l2 không thuộc clean_l1 → bỏ clean_l2
    # (xử lý đơn giản: chỉ giữ L2 nếu L1 hợp lệ)
    if not clean_l1:
        clean_l2 = ''
        clean_l3 = ''

    a['clean_l1'] = clean_l1
    a['clean_l2'] = clean_l2
    a['clean_l3'] = clean_l3

# Kiểm tra nhanh 5 bài có thay đổi
print('Ví dụ bài có nhãn thay đổi (L2 gốc → L2 mới):')
shown = 0
for a in raw:
    if a['label_l2'] != a['clean_l2'] and a['label_l2']:
        print(f'  L2: {a["label_l2"]:<25} → {a["clean_l2"]!r:<25}  '
              f'L3: {a["label_l3"]!r:<20} → {a["clean_l3"]!r}')
        shown += 1
        if shown >= 8: break


## Cell 7 — Thống kê trước/sau
So sánh số lượng nhãn và bài trước và sau khi xử lý để xác nhận kết quả.


In [ ]:
# ============================================================
# CELL 7 — Thống kê trước vs sau xử lý
# ============================================================

before_l2 = Counter(a['label_l2'] for a in raw if a['label_l2'])
after_l2  = Counter(a['clean_l2'] for a in raw if a['clean_l2'])
before_l3 = Counter(a['label_l3'] for a in raw if a['label_l3'])
after_l3  = Counter(a['clean_l3'] for a in raw if a['clean_l3'])

print(f'{'='*55}')
print(f'  TRƯỚC XỬ LÝ')
print(f'  Nhãn L2 unique: {len(before_l2):<5}  Nhãn L3 unique: {len(before_l3)}')
print(f'  Bài có L2: {sum(before_l2.values()):<7}  Bài có L3: {sum(before_l3.values())}')
print(f'  SAU XỬ LÝ')
print(f'  Nhãn L2 unique: {len(after_l2):<5}  Nhãn L3 unique: {len(after_l3)}')
print(f'  Bài có L2: {sum(after_l2.values()):<7}  Bài có L3: {sum(after_l3.values())}')
print(f'{'='*55}')

print(f'\nNHÃN L2 SAU XỬ LÝ (sắp theo tần suất):')
for name, cnt in after_l2.most_common():
    bar = '█' * (cnt * 30 // after_l2.most_common(1)[0][1])
    print(f'  {name:<28} {cnt:>5}  {bar}')

print(f'\nNHÃN L3 SAU XỬ LÝ (sắp theo tần suất):')
for name, cnt in after_l3.most_common():
    bar = '█' * (cnt * 30 // max(after_l3.values()))
    print(f'  {name:<28} {cnt:>5}  {bar}')


## Cell 8 — Build HIERARCHY từ data sạch
Xây dựng cây nhãn thực tế từ data đã xử lý. Đây là HIERARCHY sẽ dùng trong preprocessing.


In [ ]:
# ============================================================
# CELL 8 — Build HIERARCHY từ data sạch
# Đây là cấu trúc cây nhãn thực tế của dataset 5000 bài
# ============================================================

# Xây dựng từ data: L1 → {L2 → [L3]}
hierarchy_raw = defaultdict(lambda: defaultdict(set))

for a in raw:
    l1 = a['clean_l1']
    l2 = a['clean_l2']
    l3 = a['clean_l3']
    if not l1:
        continue
    if l2:
        if l3:
            hierarchy_raw[l1][l2].add(l3)
        else:
            hierarchy_raw[l1][l2]  # khởi tạo không có L3

# Chuyển set → sorted list
HIERARCHY = {
    l1: {l2: sorted(l3s) for l2, l3s in sorted(subs.items())}
    for l1, subs in sorted(hierarchy_raw.items())
}

# In cây HIERARCHY
l1_count = len(HIERARCHY)
l2_count = sum(len(subs) for subs in HIERARCHY.values())
l3_count = sum(len(l3s) for subs in HIERARCHY.values() for l3s in subs.values())

print(f'HIERARCHY: {l1_count} L1 / {l2_count} L2 / {l3_count} L3')
print()
for l1, subs in HIERARCHY.items():
    print(f'{l1} ({len(subs)} L2):')
    for l2, l3s in subs.items():
        l3_str = f'  [{', '.join(l3s)}]' if l3s else ''
        print(f'  └─ {l2}{l3_str}')
    print()


## Cell 9 — Tokenize, encode multi-hot & lưu dataset.json
Bước cuối: tokenize, tạo label_map cố định, encode multi-hot vector, build vocab, lưu file.


In [ ]:
# ============================================================
# CELL 9 — Tokenize + encode multi-hot + build vocab + lưu
# ============================================================

# ── Label map cố định từ HIERARCHY ──────────────────────────────────────
all_l1 = sorted(HIERARCHY.keys())
all_l2 = sorted({l2 for subs in HIERARCHY.values() for l2 in subs})
all_l3 = sorted({l3 for subs in HIERARCHY.values() for l3s in subs.values() for l3 in l3s})

LABEL_MAP = {
    'l1': {lb: i for i, lb in enumerate(all_l1)},
    'l2': {lb: i for i, lb in enumerate(all_l2)},
    'l3': {lb: i for i, lb in enumerate(all_l3)},
}
print(f'Label map: L1={len(LABEL_MAP["l1"])} L2={len(LABEL_MAP["l2"])} L3={len(LABEL_MAP["l3"])}')

# ── Tokenize ──────────────────────────────────────────────────────────────
import re
STOPWORDS = {
    'và','của','là','có','cho','trong','với','được','này','đó','các','một',
    'những','từ','về','đã','đến','khi','như','theo','tại','trên','hay',
    'cũng','bị','rằng','mà','thì','vì','nên','còn','sẽ','để','lại','ra',
    'vào','đây','ở','sau','trước','qua','rất','hơn','không','thế','người',
    'năm','ngày','tháng','số','ông','bà','anh','chị','em','họ','tôi','ta',
}

def clean_text(text):
    text = re.sub(r'https?://\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

def tokenize(text):
    try:
        from underthesea import word_tokenize
        tokens = word_tokenize(clean_text(text), format='text').split()
    except ImportError:
        tokens = clean_text(text).split()
    return [t for t in tokens if t not in STOPWORDS and len(t) > 1]

def make_multihot(label: str, label_to_idx: dict) -> list:
    vec = [0] * len(label_to_idx)
    if label and label in label_to_idx:
        vec[label_to_idx[label]] = 1
    return vec

# ── Tokenize + encode toàn bộ ─────────────────────────────────────────────
MIN_TOKENS = 20
output = []
skipped = 0

print('Tokenize...')
for i, a in enumerate(raw):
    l1, l2, l3 = a['clean_l1'], a['clean_l2'], a['clean_l3']
    if not l1:             # bỏ bài không xác định domain
        skipped += 1
        continue
    full_text = a.get('title','') + ' ' + a.get('content','')
    tokens = tokenize(full_text)
    if len(tokens) < MIN_TOKENS:
        skipped += 1
        continue
    output.append({
        'url':           a.get('url',''),
        'title':         a.get('title',''),
        'tokens':        tokens,
        'labels_l1':     [l1] if l1 else [],
        'labels_l2':     [l2] if l2 else [],
        'labels_l3':     [l3] if l3 else [],
        'vec_l1':        make_multihot(l1, LABEL_MAP['l1']),
        'vec_l2':        make_multihot(l2, LABEL_MAP['l2']),
        'vec_l3':        make_multihot(l3, LABEL_MAP['l3']),
        'is_multilabel': False,
    })
    if (i+1) % 500 == 0:
        print(f'  {i+1}/{len(raw)}')

print(f'\nBài hợp lệ : {len(output)}')
print(f'Bỏ qua     : {skipped}')

# ── Build vocab ───────────────────────────────────────────────────────────
from collections import Counter
counter = Counter(t for a in output for t in a['tokens'])
vocab = {'<PAD>':0,'<UNK>':1}
for word, cnt in counter.most_common():
    if cnt >= 3:
        vocab[word] = len(vocab)
print(f'Vocab size : {len(vocab):,}')

# ── Lưu file ──────────────────────────────────────────────────────────────
with open('dataset.json','w',encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
with open('vocab.json','w',encoding='utf-8') as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)
with open('label_map.json','w',encoding='utf-8') as f:
    json.dump(LABEL_MAP, f, ensure_ascii=False, indent=2)
with open('hierarchy.json','w',encoding='utf-8') as f:
    json.dump(HIERARCHY, f, ensure_ascii=False, indent=2)

print('\n✓ dataset.json  ✓ vocab.json  ✓ label_map.json  ✓ hierarchy.json')


## Cell 10 — Kiểm tra output cuối
Đọc lại dataset.json, in mẫu và thống kê cuối để xác nhận mọi thứ đúng.


In [ ]:
# ============================================================
# CELL 10 — Kiểm tra output cuối
# Đọc lại file đã lưu, in thống kê và 3 bài mẫu
# ============================================================

with open('dataset.json', encoding='utf-8') as f:
    saved = json.load(f)

total      = len(saved)
has_l2     = sum(1 for a in saved if a['labels_l2'])
has_l3     = sum(1 for a in saved if a['labels_l3'])

print(f'{'='*55}')
print(f'  THỐNG KÊ DATASET CUỐI CÙNG')
print(f'{'='*55}')
print(f'  Tổng bài      : {total:,}')
print(f'  Có L2         : {has_l2:,} ({has_l2/total*100:.1f}%)')
print(f'  Có L3         : {has_l3:,} ({has_l3/total*100:.1f}%)')
print(f'  Nhãn L1       : {len(LABEL_MAP["l1"])}')
print(f'  Nhãn L2       : {len(LABEL_MAP["l2"])}')
print(f'  Nhãn L3       : {len(LABEL_MAP["l3"])}')
print(f'  Vocab size    : {len(vocab):,}')
print(f'  vec_l1 length : {len(saved[0]["vec_l1"])}')
print(f'  vec_l2 length : {len(saved[0]["vec_l2"])}')
print(f'  vec_l3 length : {len(saved[0]["vec_l3"])}')
print(f'{'='*55}')

# Phân phối L1
from collections import defaultdict
l1_dist = defaultdict(int)
for a in saved:
    for lb in a['labels_l1']:
        l1_dist[lb] += 1
print('\nPhân phối L1:')
mx = max(l1_dist.values()) if l1_dist else 1
for lb, cnt in sorted(l1_dist.items(), key=lambda x: -x[1]):
    bar = '█' * (cnt * 25 // mx)
    print(f'  {lb:<22} {cnt:>5}  {bar}')

# 3 bài mẫu
print('\n3 bài mẫu:')
for a in saved[:3]:
    print(f"  title   : {a['title'][:55]}")
    print(f"  L1={a['labels_l1']}  L2={a['labels_l2']}  L3={a['labels_l3']}")
    print(f"  tokens  : {a['tokens'][:6]} ...")
    print(f"  vec_l1  : sum={sum(a['vec_l1'])}  vec_l2: sum={sum(a['vec_l2'])}  vec_l3: sum={sum(a['vec_l3'])}")
    print()
